# Network bargaining analysis
Rebuild the saved tables and figures from logs. Dry runs are scripted-bot checks, not study results. No API calls are made.


In [ ]:
import json, os
from pathlib import Path
PROJECT = Path.cwd()
if not (PROJECT / "pyproject.toml").exists() and (PROJECT.parent / "pyproject.toml").exists():
    PROJECT = PROJECT.parent
raw_roots = os.environ.get("LNB_ROOTS", str(PROJECT / "runs"))
ROOTS = json.loads(raw_roots) if raw_roots.lstrip().startswith("[") else raw_roots.split(os.pathsep)
OUT = Path(os.environ.get("LNB_OUT", str(PROJECT / "analysis_notebook")))
INCLUDE_DRY = os.environ.get("LNB_INCLUDE_DRY", "0").lower() in ("1", "true", "yes")
SEED, BOOT = 1, 2000
display = globals().get("display", print)


## Source tables
Episodes are the independent observations. The command also writes every Phase 6 table and the no-last-round check.


In [ ]:
from llm_network_bargaining.analysis import load_dataset
from llm_network_bargaining.analysis.cli import main as analyse
args = [*ROOTS, "--out", str(OUT / "analysis"), "--seed", str(SEED), "--boot", str(BOOT)]
if INCLUDE_DRY:
    args.append("--include-dry")
assert analyse(args) == 0
data = load_dataset(ROOTS, include_dry=INCLUDE_DRY)
display(data.episodes)


## Registered hypotheses and predictor comparisons
H2 ranks the four point benchmarks. Centrality enters ordering accuracy only.


In [ ]:
from llm_network_bargaining.analysis.hypotheses import run_hypotheses
from llm_network_bargaining.analysis.benchmarks import benchmark_errors, ordering_accuracy
from llm_network_bargaining.analysis.mixed import fit_mixed_models
display(run_hypotheses(data, seed=SEED, n_boot=BOOT).to_dict())
display(benchmark_errors(data, seed=SEED, n_boot=BOOT))
display(ordering_accuracy(data, seed=SEED, n_boot=BOOT))
display(fit_mixed_models(data))


## Figures
Episode dots accompany the means and bootstrap intervals. Human anchors are references from other protocols, not matched controls.


In [ ]:
from llm_network_bargaining.analysis.figures import (
    fig_observed_vs_predicted, fig_power_use, fig_dose_response,
    fig_parity_flip, fig_learning_curves, save_figure)
import matplotlib.pyplot as plt
for name, function in [("observed_vs_predicted", fig_observed_vs_predicted),
                       ("power_use", fig_power_use), ("dose_response", fig_dose_response),
                       ("parity_flip", fig_parity_flip), ("learning_curves", fig_learning_curves)]:
    figure = function(data, seed=SEED, n_boot=BOOT)
    save_figure(figure, OUT / "figures" / name)
    display(figure)
    plt.close(figure)


In [ ]:
for config in ("stretch_2c_sweep", "stretch_2d_control"):
    if config in set(data.episodes.config_name):
        display(benchmark_errors(data, config=config, seed=SEED, n_boot=BOOT))
        display(ordering_accuracy(data, config=config, seed=SEED, n_boot=BOOT))
        figure = fig_observed_vs_predicted(data, config=config, seed=SEED, n_boot=BOOT)
        save_figure(figure, OUT / "figures" / f"observed_vs_predicted_{config}")
        display(figure)
        plt.close(figure)


## Diagnostics and pilot gate
Failed and interrupted costs remain separate from completed-episode outcomes. The game-length decision is informational in the gate.


In [ ]:
from llm_network_bargaining.analysis.gate import pilot_gate
display(json.loads((OUT / "analysis" / "diagnostics.json").read_text()))
display(pilot_gate(data, ROOTS).to_dict())
